In [95]:
import json
from collections import Counter
from pathlib import Path

In [96]:
TEST_INPUT_PATH = Path("/home/yusuf/explainbench/logs_zhiyuan/logs/run_evaluation/trace.debug.gold.1021/gold")

In [97]:
def gather_jsonl_files(input_path: Path):
    json_files = list(input_path.rglob("*.jsonl"))
    json_files = [f for f in json_files if "patched_traces" in str(f)]
    return json_files

In [98]:
jsonl_files = gather_jsonl_files(TEST_INPUT_PATH)

In [99]:
def open_jsonl_file(file_path: Path):
    with open(file_path, "r") as f:
        for line in f:
            yield json.loads(line)
            
def filter_seen_variables(seen_variables, allowed_keys):
    """
    Filter a seen_variables dict to only keep specific top-level keys.
    """
    return {k: v for k, v in seen_variables.items() if k in allowed_keys}

from collections import Counter

def count_py_objects(obj):
    """
    Recursively walk through obj (dicts, lists, tuples, sets)
    and count occurrences of dicts that have a "py/object" key.
    Returns a Counter mapping py/object value -> count.
    """
    counter = Counter()

    def _walk(current):
        if isinstance(current, dict):
            # If this dict has "py/object", record it
            if "py/object" in current:
                counter[current["py/object"]] += 1

            # Recurse into all values
            for v in current.values():
                _walk(v)

        elif isinstance(current, (list, tuple, set)):
            for item in current:
                _walk(item)
        # primitives are ignored

    _walk(obj)
    return counter


In [100]:
from collections import Counter
from tqdm import tqdm

per_path_counts = {}

for path in tqdm(jsonl_files, desc="JSONL files"):
    path_counter = Counter()
    prev_record = None

    for idx, record in enumerate(
        tqdm(open_jsonl_file(path), desc=f"Records in {path}", leave=False)
    ):
        seen_variables = record.get("seen_variables", {})

        if seen_variables:
            if prev_record is not None:
                vars_defined = prev_record.get("vars_defined", [])
                if vars_defined:
                    seen_variables = filter_seen_variables(
                        seen_variables, vars_defined
                    )

            path_counter.update(count_py_objects(seen_variables))

        prev_record = record 
    
    project = [x for x in path.parts if "__" in x][0]
    per_path_counts[project] = path_counter


JSONL files: 100%|██████████| 42/42 [00:09<00:00,  4.41it/s]


In [101]:
per_path_counts

{'astropy__astropy-14096': Counter({'astropy.units.core.PrefixUnit': 276453,
          'astropy.units.core.CompositeUnit': 126256,
          'astropy.extern.ply.yacc.MiniProduction': 49822,
          'astropy.units.core.Unit': 26458,
          'astropy.units.core.IrreducibleUnit': 17398,
          're.Pattern': 4385,
          'astropy.extern.ply.yacc.YaccSymbol': 3363,
          'astropy.extern.ply.lex.LexToken': 2912,
          'astropy.extern.ply.lex.Lexer': 2177,
          're.Match': 1863,
          '_sitebuiltins._Printer': 1611,
          '__iterator__': 1244,
          'numpy.float64': 1119,
          '_frozen_importlib.ModuleSpec': 1074,
          '_sitebuiltins.Quitter': 1074,
          'astropy.extern.ply.yacc.LRParser': 827,
          '_frozen_importlib_external.SourceFileLoader': 784,
          'builtins.ellipsis': 537,
          'builtins.NotImplementedType': 537,
          '_sitebuiltins._Helper': 537,
          'astropy.extern.ply.yacc.YaccProduction': 426,
          'a

In [102]:
total_counts = Counter()
for counter in per_path_counts.values():
    total_counts.update(counter)

In [103]:
total_counts

Counter({'astropy.units.core.PrefixUnit': 400279,
         'astropy.units.core.CompositeUnit': 126416,
         'numpy.ndarray': 116466,
         'astropy.extern.ply.yacc.MiniProduction': 64314,
         'astropy.units.core.Unit': 32187,
         'astropy.table.column.Column': 22494,
         'astropy.units.core.IrreducibleUnit': 19954,
         'astropy.io.fits.column.Column': 17576,
         'astropy.io.fits.column._ColumnFormat': 17572,
         'astropy.time.core.Time': 16116,
         'builtins.complex': 8436,
         'numpy.dtypes.VoidDType': 8285,
         'astropy.config.configuration.ConfigItem': 8246,
         'astropy.io.fits.card.Card': 6984,
         'astropy.time.core.TimeInfo': 6441,
         're.Pattern': 5543,
         'astropy.table.table.TableColumns': 5252,
         'astropy.extern.configobj.validate.Validator': 5201,
         'astropy.extern.ply.yacc.YaccSymbol': 4817,
         'astropy.table.table.Table': 4817,
         'astropy.table.column.ColumnInfo': 4491,
  